# Wiggle Plots Demo
Plot wiggle traces with Matplotlib, Bokeh, and Plotly.

In [ ]:
%pip install matplotlib plotly bokeh segyio

# *** Install pyseiskit from PyPI
# %pip install pyseiskit
# *** Install pyseiskit for local development
# %pip install -e ..
!uv pip install -e .. --reinstall

## 1A. Option 1: Create Simple Synthetic Data
Run this cell to generate a small 2D array representing 5 seismic traces, each with 100 time samples.

In [ ]:
import numpy as np

# 100 samples, 5 traces
gatherData = np.random.randn(100, 5) 
timeSamples = np.arange(100)
traceOffsets = np.arange(5, dtype=float)


## 1B. Option 2: Load Real `.su` or `.sgy` Data
Run this cell instead if you have the file `tac-204RL239.su` in the same folder as this notebook.

In [ ]:
import segyio
from seismicReader import readSeismicFile

filename = 'tac-204RL239.su'

# We only want to plot a single gather to avoid rendering thousands of traces.
# FieldRecord corresponds to the 'fldr' header in SU/SEGY.
gatherKey = segyio.TraceField.FieldRecord
gatherIndex = 50  # Change this to a valid fldr number from your dataset!

try:
    gatherData, traceOffsets, timeSamples = readSeismicFile(filename, gatherKey=gatherKey, gatherIndex=gatherIndex)
    print(f"Loaded {gatherData.shape[1]} traces for fldr {gatherIndex} with {gatherData.shape[0]} samples each.")
except FileNotFoundError:
    print(f"File '{filename}' not found. Please place it in the 'demos' folder or stick to Option 1A.")
except ValueError as e:
    print(f"Error: {e}")

## 2. Process Data for Wiggles
Calculate the scaling and coordinates for the wiggle trace polygons.

In [ ]:
from pyseiskit import sourceData

scaledGatherData = sourceData.rescaleDataForWiggle(gatherData, traceOffsets, overlap=1.0)
lineData = sourceData.wiggleLinesDataFactory(scaledGatherData, traceOffsets, timeSamples)
patchData = sourceData.wigglePatchesDataFactory(scaledGatherData, traceOffsets, timeSamples, fill_mode='positive')

## 3. Plotting with Matplotlib

In [ ]:
import matplotlib.pyplot as plt

matplotlibFigure, matplotlibAxes = plt.subplots(figsize=(6, 4))

# Plot filled areas
for x, y in zip(patchData['xs'], patchData['ys']):
    matplotlibAxes.fill(x, y, color='black')

# Plot trace lines
for x, y in zip(lineData['xs'], lineData['ys']):
    matplotlibAxes.plot(x, y, color='black', linewidth=0.5)

matplotlibAxes.invert_yaxis() # Time goes down
plt.show()

## 4. Plotting with Bokeh

In [ ]:
from bokeh.plotting import figure, show, output_notebook
output_notebook()

bokehPlot = figure(width=600, height=400, y_range=(timeSamples[-1], timeSamples[0]))
bokehPlot.multi_line(**lineData, color='black', line_width=0.5)
bokehPlot.patches(**patchData, color='black', line_width=0)

show(bokehPlot)

## 5. Plotting with Plotly

In [ ]:
import plotly.graph_objects as go

plotlyFigure = go.Figure()

for x, y in zip(lineData['xs'], lineData['ys']):
    plotlyFigure.add_trace(go.Scatter(x=x, y=y, mode='lines', line=dict(color='black', width=0.5), showlegend=False))

for x, y in zip(patchData['xs'], patchData['ys']):
    plotlyFigure.add_trace(go.Scatter(x=x, y=y, fill='toself', fillcolor='black', line=dict(width=0), showlegend=False))

plotlyFigure.update_layout(yaxis=dict(autorange='reversed'), height=500)
plotlyFigure.show()